# Nos sujets sont-ils atrophiés ? Comparaison visuelle avec le modèle MNI

**La question.** Jusqu'ici, « atrophié » reposait sur un indicateur indirect : la part de voxels sombres (liquide) dans le masque du cerveau de la FLAIR. Ce notebook le vérifie à l'oeil, en mettant les sujets **dans le même espace que le modèle** (template MNI152NLin2009cAsym) et en comparant les coupes.

**Le modèle de référence.** MNI152NLin2009cAsym (Fonov et al., NeuroImage 2011) est la moyenne recalée de **152 IRM de jeunes adultes en bonne santé** (base ICBM). Ses ventricules sont donc ceux d'un cerveau jeune. Le challenge WMH ne publie pas l'âge des sujets (absent de `participants.tsv`) : on ne peut pas séparer l'effet de l'âge de celui de la maladie, seulement constater une différence.

**Le piège à éviter : quel recalage montrer ?**

```
sujet T1 (N4, cerveau seul)
   |
   |  affine (12 paramètres) : position, orientation, taille globale, cisaillement
   v
"espace affine"   <- les FORMES restent celles du sujet : des ventricules dilatés restent dilatés
   |
   |  SyN (déformation non linéaire locale)
   v
"espace SyN"      <- la déformation a ÉTIRÉ ou COMPRIMÉ le sujet pour ressembler au modèle :
                     si elle réussit, les ventricules ont la taille de ceux du modèle
```

- Pour **voir l'atrophie**, il faut regarder l'**espace affine** : la taille globale de la tête est normalisée, donc il ne reste que les différences de forme, en particulier la taille relative des ventricules et des sillons.
- L'**espace SyN** montre autre chose : **ce que le recalage a réussi à corriger**. Si les ventricules du sujet y restent plus grands que ceux du modèle, SyN n'a pas pu suivre. C'est ce qui explique les corrélations faibles du benchmark.
- **La quantité d'atrophie** est lisible dans la déformation elle-même : le **déterminant jacobien** de SyN dit, en chaque point du modèle, par combien la déformation multiplie le volume local (2 = la région correspondante du sujet est deux fois plus grande). C'est le principe de la morphométrie par tenseur de déformation (TBM).

**Les sujets.** Ceux qui ont déjà une transformation vers le modèle (aucun nouveau recalage n'est lancé ici) : sub-000 et sub-002 (pipeline, étape 3.6), sub-049, sub-021 et sub-053 (benchmark, variante par défaut). Le classement de l'indicateur indirect : sub-049 le moins atrophié, sub-021 et sub-000 parmi les plus atrophiés.

Coût : environ 1 min et moins de 2 Go de RAM.

In [ ]:
import ants
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

from wmh_multisite.config import load_config, resolve_path
from wmh_multisite.preproc import PIPELINE
from wmh_multisite.preproc.atlas import masked, subject_paths, template_files
from wmh_multisite.utils import io

cfg = load_config()
tpl = template_files(cfg)
D = resolve_path(cfg, "derivatives") / PIPELINE
BENCH = D / "_syn_benchmark"
SUBJECTS = ["sub-049", "sub-002", "sub-053", "sub-021", "sub-000"]   # indirect indicator: least -> most atrophic


def transforms(sub, variant=None):
    # variant=None: pipeline transforms if present (default SyN), else the default variant of the benchmark.
    # variant="mattes_100-70-50-20" etc.: that variant of the SyN benchmark.
    p = subject_paths(cfg, sub)
    if variant is None and p["affine"].is_file():
        p["source"] = "pipeline (default)"
        return p
    variant = variant or "default_mattes_40-20-0"
    for k in ("affine", "warp", "vent", "vent_atlas"):
        p[k] = BENCH / f"{sub}__{variant}" / p[k].name
    p["source"] = f"benchmark ({variant})"
    return p


t1_tpl = masked(ants.image_read(str(tpl["t1"]), pixeltype="float"), ants.image_read(str(tpl["brain"])))
brain_tpl = ants.image_read(str(tpl["brain"])).numpy() > 0.5
labels = ants.image_read(str(tpl["labels"])).numpy()
vent_tpl = np.isin(labels, cfg["preproc"]["atlas"]["ventricle_labels"])
VOX_ML = float(np.prod(t1_tpl.spacing)) / 1000
print(f"template: {t1_tpl.shape}, brain {brain_tpl.sum() * VOX_ML:.0f} ml, lateral ventricles {vent_tpl.sum() * VOX_ML:.1f} ml")
print({s: transforms(s)["source"] for s in SUBJECTS})

## 1. Mettre chaque sujet dans l'espace du modèle

Les sections 1 à 5 utilisent le **réglage SyN par défaut** pour les 5 sujets (pipeline pour sub-000 et sub-002, variante par défaut du benchmark pour les autres) ; la section 6 compare avec les réglages plus poussés.

Pour chaque sujet, trois images sur la grille du modèle (1 mm) :
- **affine** : le T1 du sujet (N4, masque HD-BET) transporté avec la seule partie affine du recalage ;
- **SyN** : le même avec la transformation complète (affine puis déformation) ;
- **jacobien** : le déterminant jacobien de la déformation SyN, calculé sur la grille du modèle.

Le facteur d'échelle de l'affine (son déterminant) donne en plus la **taille du cerveau du sujet par rapport au modèle** (1,1 = 10 % plus grand).

Pour l'affichage, chaque image est divisée par la médiane de sa substance blanche approximative (le 75e centile du cerveau). Les contrastes deviennent ainsi comparables d'un sujet à l'autre, sans changer les formes.

In [ ]:
def to_template(sub, variant=None):
    p = transforms(sub, variant)
    moving = masked(ants.image_read(str(p["t1_n4"]), pixeltype="float"), ants.image_read(str(p["t1_brain"])))
    affine_only = ants.apply_transforms(t1_tpl, moving, [str(p["affine"])], interpolator="linear")
    full = ants.apply_transforms(t1_tpl, moving, [str(p["warp"]), str(p["affine"])], interpolator="linear")
    jac_img = ants.create_jacobian_determinant_image(t1_tpl, str(p["warp"]))
    if jac_img.shape != t1_tpl.shape:  # warp computed on the 2 mm template: Jacobian back on the 1 mm grid
        jac_img = ants.resample_image_to_target(jac_img, t1_tpl, interp_type="linear")
    jac = jac_img.numpy()
    m = np.asarray(ants.read_transform(str(p["affine"])).parameters[:9]).reshape(3, 3)
    out = {"affine": affine_only.numpy(), "syn": full.numpy(), "jac": jac, "scale": abs(np.linalg.det(m)), "p": p}
    for k in ("affine", "syn"):
        img = out[k]
        out[k] = img / np.percentile(img[brain_tpl & (img > 0)], 75)
    return out


def norm_tpl(img):
    return img / np.percentile(img[brain_tpl], 75)


TPL = norm_tpl(t1_tpl.numpy())
R = {s: to_template(s) for s in SUBJECTS}
print("done:", list(R))

## 2. Coupes : le modèle, puis chaque sujet dans l'espace affine

Coupes **axiale** et **coronale** passant par le centre des ventricules latéraux du modèle. Sur chaque image :
- **contour rouge** : les ventricules latéraux du modèle (atlas Harvard-Oxford, labels 3 et 14) ;
- **contour jaune** : le masque du cerveau du modèle.

**Comment lire.** Si le sujet n'était pas atrophié, son liquide (noir en T1) remplirait à peu près le contour rouge. Un sujet atrophié montre :
1. des **ventricules qui débordent** largement du contour rouge ;
2. des **sillons corticaux élargis** (traits noirs entre les circonvolutions, près du contour jaune) ;
3. un **espace noir** entre le cerveau et le contour jaune : le cerveau occupe moins la boîte crânienne.

Convention neurologique : la gauche du sujet est à gauche de l'image.

In [ ]:
ci, cj, ck = np.round(np.argwhere(vent_tpl).mean(0)).astype(int)    # ventricle centroid (voxels)


def views(vol):
    return {"axial": vol[:, :, ck].T, "coronal": vol[:, cj, :].T}


def show(ax, img, title, mask_views=True):
    ax.imshow(img, cmap="gray", origin="lower", vmin=0, vmax=1.4)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


def contours(ax, view):
    ax.contour(views(vent_tpl.astype(float))[view], [0.5], colors="red", linewidths=0.8)
    ax.contour(views(brain_tpl.astype(float))[view], [0.5], colors="yellow", linewidths=0.6)


rows = [("modèle MNI (jeunes adultes)", TPL)] + [(f"{s} (affine)", R[s]["affine"]) for s in SUBJECTS]
fig, axes = plt.subplots(len(rows), 2, figsize=(7.5, 3.6 * len(rows)))
for (name, vol), ax_row in zip(rows, axes):
    for ax, (view, img) in zip(ax_row, views(vol).items()):
        show(ax, img, f"{name} - {view}")
        contours(ax, view)
fig.legend([Line2D([], [], color="red"), Line2D([], [], color="yellow")],
           ["ventricules latéraux du modèle", "cerveau du modèle"], loc="upper center", ncol=2)
plt.tight_layout(rect=(0, 0, 1, 0.985))
plt.show()

## 3. Même chose après SyN : ce que la déformation a corrigé

À gauche, l'espace **affine** (formes du sujet) ; à droite, l'espace **SyN** (après déformation). Si SyN avait parfaitement réussi, les ventricules de droite rempliraient exactement le contour rouge.
- Ce qui **reste en dehors** du contour rouge à droite est ce que SyN (réglage par défaut) n'a pas pu rattraper.
- C'est ce reste qui fait chuter la corrélation et la part des ventricules du modèle qui tombe sur du liquide (`tpl_vent_on_csf`) dans le benchmark.

In [ ]:
fig, axes = plt.subplots(len(SUBJECTS), 4, figsize=(14, 3.4 * len(SUBJECTS)))
for s, ax_row in zip(SUBJECTS, axes):
    for j, (space, view) in enumerate([("affine", "axial"), ("syn", "axial"), ("affine", "coronal"), ("syn", "coronal")]):
        show(ax_row[j], views(R[s][space])[view], f"{s} - {space} - {view}")
        contours(ax_row[j], view)
plt.tight_layout()
plt.show()

## 4. La déformation, carte par carte : le déterminant jacobien

Couleur sur la coupe du modèle : **rouge** = la région correspondante du sujet est **plus grande** que dans le modèle (dilatation), **bleu** = plus petite (contraction), échelle logarithmique en base 2 (+1 = deux fois plus grand, -1 = deux fois plus petit).

Ce qu'on attend d'un cerveau atrophié : du **rouge dans les ventricules** (liquide en excès) et du **bleu dans le tissu** autour (tissu perdu). Seule la partie que SyN a réussi à déformer apparaît : quand SyN n'a pas suivi (section 3), la carte **sous-estime** l'atrophie.

In [ ]:
fig, axes = plt.subplots(len(SUBJECTS), 2, figsize=(7.5, 3.4 * len(SUBJECTS)))
for s, ax_row in zip(SUBJECTS, axes):
    logj = np.log2(np.clip(R[s]["jac"], 1e-3, None)) * brain_tpl
    for ax, (view, img) in zip(ax_row, views(logj).items()):
        ax.imshow(views(TPL)[view], cmap="gray", origin="lower", vmin=0, vmax=1.4)
        im = ax.imshow(np.ma.masked_where(views(brain_tpl)[view] == 0, img), cmap="RdBu_r", origin="lower",
                       vmin=-1.5, vmax=1.5, alpha=0.8)
        ax.contour(views(vent_tpl.astype(float))[view], [0.5], colors="black", linewidths=0.6)
        ax.set_title(f"{s} - log2 jacobien - {view}", fontsize=9)
        ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.3, label="log2(volume sujet / volume modèle)")
plt.show()

## 5. En chiffres

Pour chaque sujet :
- `brain_scale` : déterminant de l'affine, soit la taille globale du cerveau du sujet par rapport au modèle. **À ne pas lire comme de l'atrophie** : le modèle MNI est plus grand qu'un cerveau moyen (1 887 ml pour son masque, contre une médiane de 1 357 ml pour nos masques HD-BET), et les deux masques ne suivent pas exactement la même définition ;
- `vent_from_jacobian_ml` : volume des ventricules du sujet **prédit par la déformation**, soit la somme, sur les ventricules du modèle, du volume local multiplié par le jacobien et par l'échelle affine. C'est une borne basse, car SyN ne suit pas tout ;
- `vent_x_template` : ce même volume rapporté à celui des ventricules du modèle ;
- `vent_refined_flair_ml` : le volume mesuré par l'ajustement sur la FLAIR (étape C de 3.6), mesure indépendante de la déformation ;
- `affine_dark_in_vent_tpl` : en espace affine, la part du contour rouge occupée par du liquide (sous 40 % de la médiane du cerveau) ;
- `affine_dark_brain_pct` : la part de liquide dans tout le masque du cerveau du modèle, en espace affine. C'est l'indicateur indirect utilisé pour choisir les sujets, recalculé ici sur une base commune. Il compte aussi l'espace vide entre le cerveau du sujet et le contour jaune (sillons élargis, cerveau plus petit que la boîte).

In [ ]:
rows = []
tpl_vent_ml = vent_tpl.sum() * VOX_ML
for s in SUBJECTS:
    r, p = R[s], R[s]["p"]
    dark = r["affine"] < 0.4
    rows.append({
        "subject": s,
        "source": p["source"],
        "brain_scale": round(r["scale"], 3),
        "vent_from_jacobian_ml": round(float(r["jac"][vent_tpl].sum()) * r["scale"] * VOX_ML, 1),
        "vent_x_template": round(float(r["jac"][vent_tpl].sum()) * r["scale"] * VOX_ML / tpl_vent_ml, 2),
        "vent_refined_flair_ml": round(io.load_labels(p["vent"], allowed=(0, 1)).sum()
                                       * np.prod(io.describe(p["vent"])["voxel_size_mm"]) / 1000, 1),
        "affine_dark_in_vent_tpl": round(float(dark[vent_tpl].mean()), 2),
        "affine_dark_brain_pct": round(100 * float(dark[brain_tpl].mean()), 1),
    })
table = pd.DataFrame(rows).set_index("subject")
print(f"template lateral ventricles: {tpl_vent_ml:.1f} ml; template dark share of brain: "
      f"{100 * (TPL[brain_tpl] < 0.4).mean():.1f} %")
table

## 6. SyN par défaut ou SyN plus poussé : la déformation suit-elle mieux ?

Les sections 1 à 5 utilisent toutes le **réglage par défaut** (itérations 40, 20, 0 : rien au niveau de résolution le plus fin). Le benchmark teste deux réglages plus poussés :
- `mattes_100-70-50-20` : même critère (information mutuelle de Mattes), mais plus d'itérations et un 4e niveau, à pleine résolution ;
- `cc_100-70-50-20` : mêmes itérations, critère de **corrélation croisée locale** (fenêtre de 4 voxels), réputé meilleur entre deux T1.
- `cc2mm_100-70-50-20` : le même CC sur le **modèle à 2 mm** (8 fois moins de voxels, rayon de 2 voxels soit ~4 mm), pour rendre 7.5 faisable sur les 170 sujets. Les images restent affichées sur la grille de 1 mm du modèle (la déformation de 2 mm est interpolée).

La cellule suivante prend, pour chaque sujet, **toutes les variantes déjà terminées** dans le benchmark. Relancer la cellule après de nouvelles fins de calcul les ajoute automatiquement.

**Ce qu'on regarde** : sur la ligne d'un sujet atrophié, est-ce que les ventricules se rapprochent du contour rouge quand on passe du défaut aux réglages poussés ? Chaque variante recalcule aussi sa propre affine : la colonne « affine » est celle de la variante par défaut.

Dans le tableau, `syn_dark_in_vent_tpl` est la part du contour rouge occupée par du liquide **après SyN** (1 = les ventricules du modèle tombent entièrement dans le liquide du sujet). `vent_from_jacobian_ml` doit se rapprocher du volume ajusté sur la FLAIR si la déformation suit mieux.

In [ ]:
ORDER = ["default_mattes_40-20-0", "mattes_100-70-50-20", "cc_100-70-50-20", "cc2mm_100-70-50-20"]
bench_csv = pd.read_csv(resolve_path(cfg, "results") / "tables" / "syn_benchmark.csv")
bench_csv = bench_csv[bench_csv.status == "ok"].set_index(["subject", "variant"])    # finished tasks only
PIPELINE_DEFAULT = {s for s in SUBJECTS if transforms(s)["source"].startswith("pipeline")}   # sub-000, sub-002
done = {s: [v for v in ORDER if (s, v) in bench_csv.index or (v == ORDER[0] and s in PIPELINE_DEFAULT)]
        for s in SUBJECTS}
CMP = {s: vs for s, vs in done.items() if len(vs) > 1}
print("subjects with several finished variants:", CMP)


def variant_arg(s, v):
    # the default SyN of sub-000 / sub-002 comes from the pipeline (step 3.6), not from the benchmark
    return None if (v == ORDER[0] and s in PIPELINE_DEFAULT) else v


RV = {(s, v): to_template(s, variant_arg(s, v)) for s, vs in CMP.items() for v in vs}

rows = []
for (s, v), r in RV.items():
    rows.append({
        "subject": s, "variant": v,
        "syn_seconds": bench_csv.syn_seconds.get((s, v)),          # None: default SyN of the pipeline
        "syn_corr_brain": bench_csv.syn_corr_brain.get((s, v)),
        "syn_dark_in_vent_tpl": round(float((r["syn"] < 0.4)[vent_tpl].mean()), 2),
        "vent_from_jacobian_ml": round(float(r["jac"][vent_tpl].sum()) * r["scale"] * VOX_ML, 1),
        "vent_refined_flair_ml": round(io.load_labels(r["p"]["vent"], allowed=(0, 1)).sum()
                                       * np.prod(io.describe(r["p"]["vent"])["voxel_size_mm"]) / 1000, 1),
    })
pd.DataFrame(rows).set_index(["subject", "variant"])

In [ ]:
n_col = 1 + max(len(vs) for vs in CMP.values())
fig, axes = plt.subplots(2 * len(CMP), n_col, figsize=(3.6 * n_col, 3.5 * 2 * len(CMP)), squeeze=False)
for i, (s, vs) in enumerate(CMP.items()):
    for k, view in enumerate(["axial", "coronal"]):
        ax_row = axes[2 * i + k]
        show(ax_row[0], views(RV[(s, vs[0])]["affine"])[view], f"{s} - affine - {view}")
        contours(ax_row[0], view)
        for j, v in enumerate(vs, 1):
            show(ax_row[j], views(RV[(s, v)]["syn"])[view], f"{s} - SyN {v.split('_')[0]} {v.split('_')[-1]} - {view}")
            contours(ax_row[j], view)
        for ax in ax_row[1 + len(vs):]:
            ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Jacobian maps, same layout: does the deformation expand the ventricles more with more iterations?
fig, axes = plt.subplots(len(CMP), n_col - 1, figsize=(3.6 * (n_col - 1), 3.5 * len(CMP)), squeeze=False)
for i, (s, vs) in enumerate(CMP.items()):
    for j, v in enumerate(vs):
        ax = axes[i, j]
        logj = np.log2(np.clip(RV[(s, v)]["jac"], 1e-3, None)) * brain_tpl
        ax.imshow(views(TPL)["axial"], cmap="gray", origin="lower", vmin=0, vmax=1.4)
        im = ax.imshow(np.ma.masked_where(views(brain_tpl)["axial"] == 0, views(logj)["axial"]), cmap="RdBu_r",
                       origin="lower", vmin=-1.5, vmax=1.5, alpha=0.8)
        ax.contour(views(vent_tpl.astype(float))["axial"], [0.5], colors="black", linewidths=0.6)
        ax.set_title(f"{s} - log2 jacobien\n{v.split('_')[0]} {v.split('_')[-1]}", fontsize=9)
        ax.axis("off")
    for ax in axes[i, len(vs):]:
        ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.4, label="log2(volume sujet / volume modèle)")
plt.show()

## 7. Retour dans l'espace du sujet : l'ajustement C sur la FLAIR

Réglage retenu pour 7.2 : **SyN par défaut + ajustement C** (J-041). Ici, plus d'espace du modèle : on regarde la **FLAIR N4 du sujet**, dans sa grille d'origine, sur 3 coupes axiales qui traversent les ventricules ajustés (bas, milieu, haut).
- **rouge** : ventricules de l'atlas transportés sur la FLAIR (étape B) ;
- **cyan** : ventricules ajustés (étape C), ceux qui serviront en 7.2. **Le cyan n'est pas « le rouge agrandi »** : C garde les voxels **sombres** (sous un seuil propre au sujet) qui sont reliés au rouge et à moins de `zone_mm` de lui. Les voxels du rouge qui tombent sur du tissu clair sont donc **retirés** (le cyan peut être plus petit que le rouge, comme chez sub-049 dont les ventricules sont fins), et le liquide sombre qui déborde du rouge est **ajouté** (le cyan est plus grand que le rouge chez les sujets atrophiés) ;
- **jaune pointillé** : limite de la bande de recherche (`zone_mm` autour des ventricules de l'atlas) ; l'ajustement ne peut pas aller au-delà.

**Ce qu'on vérifie.**
1. Le cyan suit-il le bord du liquide (noir en FLAIR) ? Trop large : il déborde sur le tissu ou les HSB ; trop étroit : il laisse du liquide dehors (cas suspect de sub-049).
2. Le cyan est-il **coupé net sur le pointillé jaune** ? Si oui, la bande est trop étroite pour ce sujet et sous-estime ses ventricules. La colonne `vent_on_band_edge_ml` du tableau le chiffre : volume ajusté situé dans la dernière tranche de la bande (épaisseur d'un voxel).

In [ ]:
import nibabel as nib
from scipy import ndimage

ZONE_MM = cfg["preproc"]["atlas"]["zone_mm"]


def canon(path):
    # RAS orientation, so that axis 2 is foot -> head whatever the acquisition (2D or 3D FLAIR)
    return nib.as_closest_canonical(nib.load(str(path)))


rows, panels, DATA = [], {}, {}
for s in SUBJECTS:
    p = transforms(s)  # default SyN (pipeline or benchmark default variant)
    img = canon(p["flair_n4"])
    zooms = img.header.get_zooms()[:3]
    flair = img.get_fdata(dtype=np.float32)
    brain = canon(p["flair_brain"]).get_fdata() > 0.5
    atlas_v = (canon(p["vent_atlas"]).get_fdata() > 0.5) & brain
    vent = canon(p["vent"]).get_fdata() > 0.5
    dist = ndimage.distance_transform_edt(~atlas_v, sampling=zooms)  # mm to the atlas ventricles
    band = brain & (dist <= ZONE_MM)
    edge = vent & (dist > ZONE_MM - max(zooms))  # last voxel-thick shell of the band
    ml = float(np.prod(zooms)) / 1000
    rows.append({"subject": s, "source": p["source"], "vent_atlas_ml": round(atlas_v.sum() * ml, 1),
                 "vent_refined_ml": round(vent.sum() * ml, 1), "vent_on_band_edge_ml": round(edge.sum() * ml, 2)})
    ks = np.where(vent.any(axis=(0, 1)))[0]
    picks = [int(np.quantile(ks, q)) for q in (0.2, 0.5, 0.8)] if ks.size else [flair.shape[2] // 2] * 3
    vmax = np.percentile(flair[brain], 99.5)
    panels[s] = [(k, flair[:, :, k].T / vmax, atlas_v[:, :, k].T, vent[:, :, k].T, band[:, :, k].T) for k in picks]
    aspect = zooms[1] / zooms[0]
    panels[s] = (panels[s], aspect)
    DATA[s] = {"flair": flair, "brain": brain, "atlas": atlas_v, "vent": vent, "zooms": zooms, "dist": dist}

pd.DataFrame(rows).set_index("subject")

In [ ]:
fig, axes = plt.subplots(len(SUBJECTS), 3, figsize=(12, 4.2 * len(SUBJECTS)))
for s, ax_row in zip(SUBJECTS, axes):
    slices, aspect = panels[s]
    for ax, (k, img, a, v, b) in zip(ax_row, slices):
        ax.imshow(img, cmap="gray", origin="lower", vmin=0, vmax=1, aspect=aspect)
        for m, color, style in [(a, "red", "solid"), (v, "cyan", "solid"), (b, "yellow", "dashed")]:
            if m.any():
                ax.contour(m.astype(float), [0.5], colors=color, linewidths=0.8, linestyles=style)
        ax.set_title(f"{s} - FLAIR coupe {k}", fontsize=9)
        ax.axis("off")
fig.legend([Line2D([], [], color="red"), Line2D([], [], color="cyan"), Line2D([], [], color="yellow", ls="--")],
           ["ventricules de l'atlas (B)", "ventricules ajustés (C)", f"limite de la bande ({ZONE_MM} mm)"],
           loc="upper center", ncol=3)
plt.tight_layout(rect=(0, 0, 1, 0.99))
plt.show()

## 8. Prototype : corriger C pour les grands ventricules (« C2 »)

**Le problème vu en section 7.** Chez les sujets très atrophiés, le rouge est beaucoup trop petit. C ne peut s'étendre qu'à 10 mm du rouge : il s'arrête sur le pointillé jaune et **oublie une partie des vrais ventricules**. Élargir simplement la bande ne marche pas : C déborde alors dans d'autres liquides (citernes autour du tronc cérébral, fissure entre les deux hémisphères), même chez les sujets sans problème.

**L'idée.** Les ventricules dilatés sont de **grosses poches** de liquide ; les débordements passent par des **canaux fins** (quelques mm de large). On s'autorise à aller plus loin, mais seulement à travers du liquide épais.

```
étape 1  candidats   : voxels sombres (même seuil que C), à moins de 30 mm du rouge,
                       et à plus de 16 mm du bord du cerveau (le liquide des sillons est au bord)
étape 2  épaissir    : ouverture morphologique dans le plan de coupe, disque de 3 mm de rayon
                       = on « gomme » tout ce qui fait moins d'environ 6 mm de large
                       (canaux fins, fissure médiane) ; les grosses poches survivent
étape 3  relier      : on garde les poches épaisses qui TOUCHENT le cyan actuel (C)
étape 4  bords       : l'ouverture a un peu rogné les bords des poches gardées : on les rend
                       (dilatation de 3 mm, limitée aux candidats de l'étape 1)
résultat C2          = C  +  poches ajoutées, trous bouchés
```

C n'est jamais diminué : C2 ne fait qu'**ajouter** des poches épaisses connectées. Chez un sujet où C est déjà bon, rien ne devrait être ajouté.

Ce prototype vit dans ce notebook seulement ; rien n'est modifié dans le pipeline (`atlas.py`) tant qu'il n'est pas validé.

In [ ]:
C2 = {"band_mm": 30, "min_depth_mm": 16, "open_mm": 3}   # prototype parameters


def disk(zooms, radius_mm):
    # in-plane disk (axes 0 and 1 after reorientation), one slice thick
    rx, ry = (int(np.ceil(radius_mm / z)) for z in zooms[:2])
    x, y = np.mgrid[-rx:rx + 1, -ry:ry + 1]
    return ((x * zooms[0]) ** 2 + (y * zooms[1]) ** 2 <= radius_mm ** 2)[:, :, None]


def c_threshold(d):
    # the same subject-specific threshold as step C (atlas.refine_ventricles)
    a = cfg["preproc"]["atlas"]
    inside = ndimage.distance_transform_edt(d["atlas"], sampling=d["zooms"])
    core = d["atlas"] & (inside > a["core_mm"])
    core = core if core.sum() >= 50 else d["atlas"]
    tissue = d["brain"] & (d["dist"] > a["tissue_min_mm"])
    return (np.median(d["flair"][core]) + np.median(d["flair"][tissue])) / 2


def c2(d):
    depth = ndimage.distance_transform_edt(d["brain"], sampling=d["zooms"])
    candidates = d["brain"] & (d["flair"] < c_threshold(d)) & (d["dist"] <= C2["band_mm"]) & (depth > C2["min_depth_mm"])
    se = disk(d["zooms"], C2["open_mm"])
    thick = ndimage.binary_opening(candidates, structure=se)                       # step 2
    lab, _ = ndimage.label(thick)
    keep = np.unique(lab[thick & ndimage.binary_dilation(d["vent"], structure=np.ones((3, 3, 3)))])
    pockets = np.isin(lab, keep[keep > 0])                                          # step 3
    pockets = ndimage.binary_dilation(pockets, structure=se) & candidates           # step 4
    final = ndimage.binary_fill_holes(d["vent"] | pockets) & d["brain"]
    return {"candidates": candidates, "thick": thick, "pockets": pockets, "final": final}


R2 = {s: c2(DATA[s]) for s in SUBJECTS}
rows = []
for s in SUBJECTS:
    ml = float(np.prod(DATA[s]["zooms"])) / 1000
    rows.append({"subject": s, "C_ml": round(DATA[s]["vent"].sum() * ml, 1),
                 "C2_ml": round(R2[s]["final"].sum() * ml, 1),
                 "added_ml": round((R2[s]["final"] & ~DATA[s]["vent"]).sum() * ml, 1)})
pd.DataFrame(rows).set_index("subject")

### 8a. Les 4 étapes, une coupe par sujet

Pour chaque sujet, la coupe où C2 ajoute le plus (ou une coupe centrale s'il n'ajoute rien). De gauche à droite :
1. **candidats** (jaune) : tout le liquide sombre autorisé, avec C en cyan ;
2. **après l'ouverture** (vert) : ce qui reste une fois les parties fines gommées ;
3. **poches gardées** (magenta) : les parties épaisses qui touchent C, bords rendus ;
4. **résultat** : C (cyan) et ce que C2 ajoute (magenta), sur la FLAIR seule.

Ce qu'il faut regarder : sur le panneau 1, le jaune contient des canaux fins (fissure, citernes) ; sur le panneau 2, ils doivent avoir disparu ; sur le panneau 4, le magenta doit être du **ventricule** (prolongement du noir déjà en cyan), pas autre chose.

In [ ]:
def overlay(ax, img, aspect, masks, title):
    ax.imshow(img, cmap="gray", origin="lower", vmin=0, vmax=1, aspect=aspect)
    for m, color, fill in masks:
        if m.any():
            if fill:
                ax.imshow(np.ma.masked_where(~m, m), cmap=matplotlib.colors.ListedColormap([color]), origin="lower",
                          alpha=0.45, aspect=aspect)
            else:
                ax.contour(m.astype(float), [0.5], colors=color, linewidths=0.8)
    ax.set_title(title, fontsize=8)
    ax.axis("off")


import matplotlib

fig, axes = plt.subplots(len(SUBJECTS), 4, figsize=(16, 4.2 * len(SUBJECTS)))
for s, ax_row in zip(SUBJECTS, axes):
    d, r = DATA[s], R2[s]
    added = r["final"] & ~d["vent"]
    k = int(np.argmax(added.sum(axis=(0, 1)))) if added.any() else int(np.median(np.where(d["vent"].any(axis=(0, 1)))[0]))
    img = d["flair"][:, :, k].T / np.percentile(d["flair"][d["brain"]], 99.5)
    aspect = d["zooms"][1] / d["zooms"][0]
    sl = lambda m: m[:, :, k].T
    overlay(ax_row[0], img, aspect, [(sl(r["candidates"]), "yellow", True), (sl(d["vent"]), "cyan", False)], f"{s} coupe {k} - 1. candidats")
    overlay(ax_row[1], img, aspect, [(sl(r["thick"]), "lime", True), (sl(d["vent"]), "cyan", False)], "2. après ouverture (épais, C inclus)")
    overlay(ax_row[2], img, aspect, [(sl(r["pockets"] & ~d["vent"]), "magenta", True), (sl(d["vent"]), "cyan", False)], "3. poches gardées, hors C")
    overlay(ax_row[3], img, aspect, [(sl(d["vent"]), "cyan", False), (sl(added), "magenta", False)], "4. résultat : C (cyan) + ajout (magenta)")
plt.tight_layout()
plt.show()

### 8b. Résultat C2 sur plusieurs coupes

Mêmes 3 coupes que la section 7 : contour cyan = C actuel, **aplat magenta** = ce que C2 ajoute, jaune pointillé = la limite actuelle de 10 mm.

In [ ]:
fig, axes = plt.subplots(len(SUBJECTS), 3, figsize=(12, 4.2 * len(SUBJECTS)))
for s, ax_row in zip(SUBJECTS, axes):
    d, r = DATA[s], R2[s]
    added = r["final"] & ~d["vent"]
    slices, aspect = panels[s]
    for ax, (k, img, a, v, b) in zip(ax_row, slices):
        ax.imshow(img, cmap="gray", origin="lower", vmin=0, vmax=1, aspect=aspect)
        add_k = added[:, :, k].T
        if add_k.any():  # filled, so that the added zone cannot be confused with a contour
            ax.imshow(np.ma.masked_where(~add_k, add_k), cmap=matplotlib.colors.ListedColormap(["magenta"]),
                      origin="lower", alpha=0.5, aspect=aspect)
        for m, color, style in [(v, "cyan", "solid"), (b, "yellow", "dashed")]:
            if m.any():
                ax.contour(m.astype(float), [0.5], colors=color, linewidths=0.8, linestyles=style)
        ax.set_title(f"{s} - coupe {k}", fontsize=9)
        ax.axis("off")
fig.legend([Line2D([], [], color="cyan"), Line2D([], [], color="magenta"), Line2D([], [], color="yellow", ls="--")],
           ["C actuel (contour)", "ajout de C2 (aplat)", f"limite actuelle ({ZONE_MM} mm)"], loc="upper center", ncol=3)
plt.tight_layout(rect=(0, 0, 1, 0.99))
plt.show()

### 8c. Les fuites de C2, montrées explicitement

Deux indicateurs simples, calculés sur ce que C2 ajoute :
- `added_midline_ml` : volume ajouté à moins de 4 mm du plan médian du cerveau. Les ventricules latéraux sont de part et d'autre de la ligne médiane (séparés par le septum) ; ce qui est ajouté **sur** la ligne médiane est surtout la fissure entre les hémisphères, le 3e ventricule ou les citernes ;
- `added_outside_C_slices_ml` : volume ajouté sur des coupes où C n'a **aucun** voxel (au-dessus ou au-dessous des ventricules) : par construction, ce n'est pas un prolongement latéral des ventricules déjà trouvés.

Puis, pour chaque sujet où C2 ajoute quelque chose : une coupe **sagittale** au milieu (le plan qui sépare les deux hémisphères, où les fuites se voient de profil), la coupe axiale **au-dessus** des ventricules où l'ajout est maximal, et la coupe axiale **basse** (sous le corps des ventricules) où il est maximal. Aplat magenta = ajout de C2, contour cyan = C.

In [ ]:
def midline_index(d):
    # x of the brain centroid (canonical RAS: axis 0 = left -> right)
    return int(round(np.argwhere(d["brain"])[:, 0].mean()))


rows = []
for s in SUBJECTS:
    d, added = DATA[s], R2[s]["final"] & ~DATA[s]["vent"]
    ml = float(np.prod(d["zooms"])) / 1000
    xm, half = midline_index(d), int(np.ceil(4 / d["zooms"][0]))
    c_slices = d["vent"].any(axis=(0, 1))
    rows.append({"subject": s, "added_ml": round(added.sum() * ml, 1),
                 "added_midline_ml": round(added[xm - half:xm + half + 1].sum() * ml, 1),
                 "added_outside_C_slices_ml": round(added[:, :, ~c_slices].sum() * ml, 1)})
pd.DataFrame(rows).set_index("subject")

In [ ]:
LEAKY = [s for s in SUBJECTS if (R2[s]["final"] & ~DATA[s]["vent"]).sum() * np.prod(DATA[s]["zooms"]) / 1000 > 1]
fig, axes = plt.subplots(len(LEAKY), 3, figsize=(15, 5 * len(LEAKY)), squeeze=False)
for s, ax_row in zip(LEAKY, axes):
    d = DATA[s]
    added = R2[s]["final"] & ~d["vent"]
    vmax = np.percentile(d["flair"][d["brain"]], 99.5)
    zx, zy, zz = d["zooms"]
    xm, half = midline_index(d), int(np.ceil(4 / zx))
    x = xm - half + int(np.argmax(added[xm - half:xm + half + 1].sum(axis=(1, 2))))
    c_k = np.where(d["vent"].any(axis=(0, 1)))[0]
    per_k = added.sum(axis=(0, 1))
    k_top = int(c_k.max() - 3 + np.argmax(per_k[c_k.max() - 3:]))                       # top of the ventricles and above
    k_low = int(c_k.min() + np.argmax(per_k[c_k.min():c_k.min() + max(1, len(c_k) // 3)]))  # lowest third
    views_ = [("sagittale médiane (x=%d)" % x, d["flair"][x].T, added[x].T, d["vent"][x].T, zz / zy),
              ("axiale haute (coupe %d)" % k_top, d["flair"][:, :, k_top].T, added[:, :, k_top].T, d["vent"][:, :, k_top].T, zy / zx),
              ("axiale basse (coupe %d)" % k_low, d["flair"][:, :, k_low].T, added[:, :, k_low].T, d["vent"][:, :, k_low].T, zy / zx)]
    for ax, (title, img, a, v, asp) in zip(ax_row, views_):
        ax.imshow(img / vmax, cmap="gray", origin="lower", vmin=0, vmax=1, aspect=asp)
        if a.any():
            ax.imshow(np.ma.masked_where(~a, a), cmap=matplotlib.colors.ListedColormap(["magenta"]), origin="lower",
                      alpha=0.55, aspect=asp)
        if v.any():
            ax.contour(v.astype(float), [0.5], colors="cyan", linewidths=0.8)
        ax.set_title(f"{s} - {title}", fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Verdict (à remplir après lecture des figures)

- Les ventricules débordent-ils visiblement du contour rouge en espace affine ? Pour quels sujets ?
- Le classement visuel suit-il l'indicateur indirect (sub-049 le moins atrophié, sub-021 et sub-000 les plus atrophiés) ?
- Le volume prédit par la déformation et le volume ajusté sur la FLAIR sont-ils du même ordre ? L'écart mesure ce que SyN ne rattrape pas.
- Pour sub-049 : la réduction de 7,4 à 3,6 ml lors de l'ajustement est-elle crédible au vu des coupes ? (section 7)
- Section 7 : un ventricule ajusté est-il coupé net sur la limite de la bande ? Pour quels sujets ?
- Section 8 : C2 récupère-t-il les ventricules oubliés sans déborder (citernes, fissure médiane) ? Ne change-t-il rien chez les sujets où C était bon ?
- Section 6 : le SyN poussé rapproche-t-il les ventricules des sujets atrophiés du contour rouge ? Le gain justifie-t-il un temps de calcul environ 7 fois plus long ?